# MICS DDML analysis

This notebook contains the main MICS double/debiased machine-learning analysis. It uses PSU-grouped cross-fitting, a convex Super Learner for the nuisance functions, raw out-of-fold propensity diagnostics, support checks by country and PSU, support-restricted robustness analyses, and leave-one-country-out diagnostics.

The principal estimates use the convex Super Learner. Estimates obtained from individual nuisance learners are retained only as optional appendix robustness checks.


## 1. Setup

The treatment names follow the Stata cleaning file:

- **No treatment**
- **Boiling**
- **Chlorination/tablets**
- **Straining/settling**

Households reporting only **Other** are removed before defining either the binary or categorical treatment. If `Other` is selected together with a recognized method, the recognized method takes precedence. Solar treatment remains a recognized treatment in the binary any-treatment analysis but is not estimated as a separate APOS category.


In [ ]:
from pathlib import Path
import json
import os
import pickle
from warnings import filterwarnings

ROOT = Path("../../").resolve()
DATA = ROOT / "Data" / "3. Final"
OUT = ROOT / "Output"
FIGS = ROOT / "Figures"
TABLES = ROOT / "Writing edit" / "Table"
MODELS = OUT / "models" / "grouped_convex_sl"
LOCO_MODELS = MODELS / "loco"
SUPPORT_MODELS = MODELS / "support_restricted"

for folder in [OUT, FIGS, TABLES, MODELS, LOCO_MODELS, SUPPORT_MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

HH_FILE = DATA / "MASTER_MICS_FINAL.dta"
U5_FILE = DATA / "MASTER_MICS_FINAL_U5.dta"

if not HH_FILE.is_file() or not U5_FILE.is_file():
    raise FileNotFoundError("The HH or U5 source file is missing.")

for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[variable] = "1"

os.environ["MPLCONFIGDIR"] = "/tmp/mics_ddml_matplotlib"
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
filterwarnings("ignore")
print(ROOT)


In [ ]:
import doubleml as dml
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import sklearn

from IPython.display import display
from joblib import hash as joblib_hash
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import (
    ElasticNetCV,
    LassoCV,
    LinearRegression,
    LogisticRegression,
    LogisticRegressionCV,
    RidgeCV,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

try:
    from doubleml.utils import PSProcessorConfig
except ImportError:
    PSProcessorConfig = None


In [ ]:
SEED = 42
FOLDS = 5
IRM_REPS = 3
APOS_REPS = 1
ROBUSTNESS_REPS = 1
LOCO_REPS = 1
TRIM = 0.01

RUN_MAIN = True
RUN_SUPPORT_ROBUSTNESS = True
RUN_LOCO = True
RUN_BASE_LEARNER_ROBUSTNESS = False

LEVELS = {
    0: "No treatment",
    1: "Boiling",
    2: "Chlorination/tablets",
    3: "Straining/settling",
}

OUTCOME_LABELS = {
    "SomeRiskHome": "Any detectable E. coli at home",
    "VeryHighRiskHome": "Very high E. coli at home (>100 CFU/100 mL)",
    "diarrhea": "Diarrhea among children under five",
}

CPU = os.cpu_count() or 1
WORKERS = max(1, CPU - 1)
INNER_FOLDS = 3

print({
    "CPU": CPU,
    "FOLDS": FOLDS,
    "IRM_REPS": IRM_REPS,
    "APOS_REPS": APOS_REPS,
    "TRIM": TRIM,
})


## 2. Data preparation

`SomeRiskHome` is an indicator for **any detectable E. coli** and therefore includes the very-high-risk observations. `VeryHighRiskHome` identifies the subset with more than 100 CFU/100 mL. Both outcomes are estimated on the same household sample whenever `RiskHome` and the covariates are observed.


In [ ]:
def _selected(series, letter):
    """Return whether a MICS multiple-response indicator selected its letter."""
    return series.astype("string").str.strip().str.upper().eq(letter.upper())


def _first_available(df, names):
    """Coalesce the first available columns in `names`."""
    result = pd.Series(pd.NA, index=df.index, dtype="object")
    found = []
    for name in names:
        if name in df.columns:
            found.append(name)
            result = result.where(result.notna(), df[name])
    if not found:
        raise KeyError(f"None of the required identifier columns exists: {names}")
    return result, found


def _factorize_key(frame):
    key = frame.astype("string").fillna("<missing>")
    return pd.factorize(pd.MultiIndex.from_frame(key), sort=True)[0].astype("int64")


def add_design_ids(df):
    """Create globally valid PSU and household identifiers and report whether country was needed."""
    df = df.copy()
    country = df["country_cat"].astype("string").fillna("<missing-country>")

    psu_raw, psu_sources = _first_available(df, ["PSU", "psu", "Cluster_var", "HH1"])
    psu_country_count = (
        pd.DataFrame({"psu": psu_raw.astype("string"), "country": country})
        .dropna()
        .groupby("psu")["country"]
        .nunique()
    )
    psu_reused = bool((psu_country_count > 1).any())
    psu_key = pd.DataFrame({"country": country, "psu": psu_raw}) if psu_reused else pd.DataFrame({"psu": psu_raw})
    df["_psu_id"] = _factorize_key(psu_key)

    if "HHID" in df.columns:
        hh_raw = df["HHID"]
        hh_sources = ["HHID"]
    elif {"HH1", "HH2"}.issubset(df.columns):
        hh_raw = df["HH1"].astype("string") + "|" + df["HH2"].astype("string")
        hh_sources = ["HH1", "HH2"]
    else:
        hh_raw, hh_sources = _first_available(df, ["HH2", "HH1"])

    hh_country_count = (
        pd.DataFrame({"hh": hh_raw.astype("string"), "country": country})
        .dropna()
        .groupby("hh")["country"]
        .nunique()
    )
    hh_reused = bool((hh_country_count > 1).any())
    hh_key = pd.DataFrame({"country": country, "hh": hh_raw}) if hh_reused else pd.DataFrame({"hh": hh_raw})
    df["_hh_id"] = _factorize_key(hh_key)

    diagnostics = {
        "psu_source_columns": ", ".join(psu_sources),
        "psu_reused_across_countries": psu_reused,
        "household_source_columns": ", ".join(hh_sources),
        "household_reused_across_countries": hh_reused,
    }
    return df, diagnostics


In [ ]:
def create_outcomes(df):
    """Create nested E. coli outcomes while preserving missing values."""
    df = df.copy()
    risk = pd.to_numeric(df["RiskHome"], errors="coerce")
    missing = risk.isna()

    df["SomeRiskHome"] = risk.isin([1, 2]).mask(missing).astype("Int8")
    df["VeryHighRiskHome"] = risk.eq(2).mask(missing).astype("Int8")

    violation = df["VeryHighRiskHome"].eq(1) & ~df["SomeRiskHome"].eq(1)
    assert not violation.any(), "Every very-high-risk observation must also be SomeRiskHome=1."
    assert df["SomeRiskHome"].isna().equals(df["VeryHighRiskHome"].isna())
    return df


def create_treatments(df):
    """Create recognized treatment categories and remove Other-only observations later."""
    df = df.copy()
    method_cols = ["WQ15A", "WQ15B", "WQ15C", "WQ15D", "WQ15E", "WQ15F", "WQ15G", "WQ15H", "WQ15X"]
    missing = sorted(set(method_cols) - set(df.columns))
    if missing:
        raise KeyError(f"Missing water-treatment indicators: {missing}")

    used = pd.DataFrame(
        {column: _selected(df[column], column[-1]) for column in method_cols},
        index=df.index,
    )

    boil = used["WQ15A"]
    chlorination = used["WQ15B"] | used["WQ15G"] | used["WQ15H"]
    straining_settling = used["WQ15C"] | used["WQ15D"] | used["WQ15F"]
    solar = used["WQ15E"]
    other = used["WQ15X"]

    recognized_any = boil | chlorination | straining_settling | solar
    main_method = boil | chlorination | straining_settling
    other_only = other & ~recognized_any
    no_treatment = ~used.any(axis=1)

    category = pd.Series(pd.NA, index=df.index, dtype="Int8")
    category.loc[no_treatment] = 0
    category.loc[straining_settling] = 3
    category.loc[chlorination] = 2
    category.loc[boil] = 1

    df["treatment_count"] = used.sum(axis=1).astype("int16")
    df["multiple_methods"] = df["treatment_count"].gt(1)
    df["other_only"] = other_only
    df["solar_treatment"] = solar
    df["water_treatment"] = recognized_any.astype("int8")
    df["treat_cat"] = category
    df["treat_boil"] = category.eq(1).fillna(False).astype("int8")
    df["treat_chlorination_tablets"] = category.eq(2).fillna(False).astype("int8")
    df["treat_straining_settling"] = category.eq(3).fillna(False).astype("int8")

    # Other-only is excluded, not recoded as untreated.
    df = df.loc[~df["other_only"]].copy()

    assert not df["other_only"].any()
    assert set(df["treat_cat"].dropna().unique()).issubset(set(LEVELS))
    return df


In [ ]:
def prepare_sample(raw, child=False):
    """Prepare the HH or U5 data without constructing sample-specific country dummies."""
    required = [
        "country_cat", "RiskHome", "windex5", "urban", "WS1_g", "wq27_decile",
        "Any_U5", "Girls_less_than15", "Boys_15or_less", "Toilet", "HHCHILDREN",
    ]
    missing = sorted(set(required) - set(raw.columns))
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df = raw.copy()
    df["_row_id"] = np.arange(len(df), dtype="int64")
    df = create_outcomes(df)
    df = create_treatments(df)
    df, id_diagnostics = add_design_ids(df)

    for column in ["Any_U5", "Girls_less_than15", "Boys_15or_less"]:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0).astype("int8")

    df["num_children"] = pd.to_numeric(df["HHCHILDREN"], errors="coerce").fillna(0).clip(upper=10)

    if child:
        needed = ["age", "male", "diarrhea"]
        missing = sorted(set(needed) - set(df.columns))
        if missing:
            raise KeyError(f"Missing U5 columns: {missing}")
        df["child_age"] = pd.to_numeric(df["age"], errors="coerce")
        df["child_sex_male"] = pd.to_numeric(df["male"], errors="coerce")
        df["diarrhea"] = pd.to_numeric(df["diarrhea"], errors="coerce").astype("Int8")

    return df, id_diagnostics


hh_raw, _ = pyreadstat.read_dta(HH_FILE)
u5_raw, _ = pyreadstat.read_dta(U5_FILE)

hh, hh_id_diagnostics = prepare_sample(hh_raw, child=False)
u5, u5_id_diagnostics = prepare_sample(u5_raw, child=True)

display(pd.DataFrame([
    {"sample": "HH", **hh_id_diagnostics},
    {"sample": "U5", **u5_id_diagnostics},
]))


In [ ]:
def _dummy_block(series, prefix, drop_first=True):
    clean = series.astype("string").fillna("Missing")
    return pd.get_dummies(clean, prefix=prefix, drop_first=drop_first, dtype=float)


def build_controls(df, child=False):
    """Build controls after each sample restriction so country dummies are regenerated in LOCO."""
    blocks = [
        _dummy_block(df["windex5"], "wealth", drop_first=True),
        _dummy_block(df["country_cat"], "country", drop_first=True),
        _dummy_block(df["urban"], "urban", drop_first=True),
        _dummy_block(df["WS1_g"], "water_source", drop_first=True),
        _dummy_block(df["Toilet"], "toilet", drop_first=True),
        _dummy_block(df["wq27_decile"], "source_ecoli", drop_first=True),
        df[["Any_U5", "Girls_less_than15", "Boys_15or_less"]].astype(float),
    ]
    if child:
        blocks.extend([
            _dummy_block(df["child_age"], "child_age", drop_first=True),
            df[["child_sex_male"]].astype(float).fillna(0),
        ])

    X = pd.concat(blocks, axis=1)
    X = X.loc[:, ~X.columns.duplicated()].astype(float)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X


def analysis_frame(df, outcome, treatment, child=False, allowed_levels=None):
    """Create a complete-case causal-model frame and its model covariates."""
    keep = df[outcome].notna() & df[treatment].notna()
    if allowed_levels is not None:
        keep &= df[treatment].isin(allowed_levels)

    sample = df.loc[keep].copy()
    X = build_controls(sample, child=child)

    frame = pd.concat([
        sample[["_row_id", "country_cat", "_psu_id", "_hh_id", outcome, treatment]],
        X,
    ], axis=1)
    frame[outcome] = pd.to_numeric(frame[outcome], errors="raise").astype(float)
    frame[treatment] = pd.to_numeric(frame[treatment], errors="raise").astype(int)

    # This column is supplied only to the custom Super Learner so its inner CV can also group by PSU.
    frame["_psu_model_code"] = pd.factorize(frame["_psu_id"], sort=True)[0].astype(float)
    x_cols = list(X.columns) + ["_psu_model_code"]
    frame = frame.reset_index(drop=True)
    return frame, x_cols


## 3. PSU and household structure

The PSU is the indivisible unit in cross-fitting: all observations from a PSU are assigned together to training or test. Household identifiers are audited as a nested no-leakage check. Country is added to the PSU key only when the raw PSU code is reused across countries.


In [ ]:
def psu_structure(df, sample_name):
    sizes = df.groupby("_psu_id").size()
    hh_sizes = df.groupby("_hh_id").size()
    return {
        "sample": sample_name,
        "observations": len(df),
        "PSUs": int(df["_psu_id"].nunique()),
        "households": int(df["_hh_id"].nunique()),
        "mean observations per PSU": sizes.mean(),
        "median observations per PSU": sizes.median(),
        "p90 observations per PSU": sizes.quantile(0.90),
        "max observations per PSU": sizes.max(),
        "mean observations per household": hh_sizes.mean(),
        "median observations per household": hh_sizes.median(),
    }


psu_structure_table = pd.DataFrame([
    psu_structure(hh, "HH"),
    psu_structure(u5, "U5"),
])
psu_structure_table.to_csv(OUT / "psu_structure_summary.csv", index=False)
display(psu_structure_table)


## 4. Convex Super Learner

The nuisance functions are convex combinations of the base learners. The meta-weights are constrained to be nonnegative and to sum to one. For the outcome nuisance, weights minimize grouped out-of-fold squared error. For the propensity nuisance, weights minimize grouped out-of-fold Bernoulli log loss.

The final learner is therefore not ridge or lasso. Ridge and lasso remain members of the base library.


In [ ]:
def _feature_and_groups(X, group_column):
    array = np.asarray(X, dtype=float)
    index = group_column if group_column >= 0 else array.shape[1] + group_column
    if index < 0 or index >= array.shape[1]:
        raise IndexError("Invalid PSU group-column index.")
    groups = array[:, index].astype("int64")
    features = np.delete(array, index, axis=1)
    return features, groups


def _grouped_inner_splits(y, groups, n_splits, random_state, classification):
    unique_groups = np.unique(groups)
    max_splits = min(int(n_splits), len(unique_groups))
    if max_splits < 2:
        raise ValueError("At least two PSUs are required for inner Super Learner cross-validation.")

    for k in range(max_splits, 1, -1):
        if classification:
            splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=random_state)
            candidate = list(splitter.split(np.zeros(len(y)), y, groups))
            valid = all(np.unique(y[train]).size == 2 for train, _ in candidate)
        else:
            splitter = GroupKFold(n_splits=k)
            candidate = list(splitter.split(np.zeros(len(y)), y, groups))
            valid = True
        if valid:
            return candidate
    raise ValueError("Could not construct grouped inner folds containing both treatment classes.")


def _positive_probability(model, X):
    probabilities = model.predict_proba(X)
    classes = np.asarray(model.classes_)
    if 1 not in classes:
        raise ValueError("A propensity learner was fitted without the positive class.")
    return probabilities[:, int(np.where(classes == 1)[0][0])]


def _convex_weights(predictions, y, classification):
    n_learners = predictions.shape[1]
    initial = np.repeat(1.0 / n_learners, n_learners)

    if classification:
        def objective(weights):
            probability = np.clip(predictions @ weights, 1e-6, 1 - 1e-6)
            return -np.mean(y * np.log(probability) + (1 - y) * np.log(1 - probability))
    else:
        def objective(weights):
            residual = y - predictions @ weights
            return np.mean(residual ** 2)

    result = minimize(
        objective,
        initial,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_learners,
        constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1.0},
        options={"maxiter": 2_000, "ftol": 1e-12},
    )

    if result.success and np.isfinite(result.fun):
        weights = np.clip(result.x, 0, 1)
        return weights / weights.sum()

    losses = []
    for j in range(n_learners):
        single = predictions[:, j]
        if classification:
            single = np.clip(single, 1e-6, 1 - 1e-6)
            loss = -np.mean(y * np.log(single) + (1 - y) * np.log(1 - single))
        else:
            loss = np.mean((y - single) ** 2)
        losses.append(loss)
    weights = np.zeros(n_learners)
    weights[int(np.argmin(losses))] = 1.0
    return weights


In [ ]:
class ConvexSuperLearnerRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, estimators, cv=3, random_state=42, group_column=-1):
        self.estimators = estimators
        self.cv = cv
        self.random_state = random_state
        self.group_column = group_column

    def fit(self, X, y):
        features, groups = _feature_and_groups(X, self.group_column)
        y = np.asarray(y, dtype=float)
        splits = _grouped_inner_splits(y, groups, self.cv, self.random_state, classification=False)

        oof = np.full((len(y), len(self.estimators)), np.nan)
        for j, (_, estimator) in enumerate(self.estimators):
            for train, test in splits:
                fitted = clone(estimator).fit(features[train], y[train])
                oof[test, j] = fitted.predict(features[test])
        if np.isnan(oof).any():
            raise RuntimeError("Incomplete outcome Super Learner OOF predictions.")

        self.weights_ = _convex_weights(oof, y, classification=False)
        self.learner_names_ = [name for name, _ in self.estimators]
        self.models_ = [clone(estimator).fit(features, y) for _, estimator in self.estimators]
        self.n_features_in_ = np.asarray(X).shape[1]
        return self

    def predict(self, X):
        features, _ = _feature_and_groups(X, self.group_column)
        predictions = np.column_stack([model.predict(features) for model in self.models_])
        return predictions @ self.weights_


class ConvexSuperLearnerClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimators, cv=3, random_state=42, group_column=-1):
        self.estimators = estimators
        self.cv = cv
        self.random_state = random_state
        self.group_column = group_column

    def fit(self, X, y):
        features, groups = _feature_and_groups(X, self.group_column)
        y = np.asarray(y, dtype=int)
        if np.unique(y).size != 2:
            raise ValueError("The propensity Super Learner requires both treatment classes.")
        splits = _grouped_inner_splits(y, groups, self.cv, self.random_state, classification=True)

        oof = np.full((len(y), len(self.estimators)), np.nan)
        for j, (_, estimator) in enumerate(self.estimators):
            for train, test in splits:
                fitted = clone(estimator).fit(features[train], y[train])
                oof[test, j] = _positive_probability(fitted, features[test])
        if np.isnan(oof).any():
            raise RuntimeError("Incomplete propensity Super Learner OOF predictions.")

        self.weights_ = _convex_weights(oof, y, classification=True)
        self.learner_names_ = [name for name, _ in self.estimators]
        self.models_ = [clone(estimator).fit(features, y) for _, estimator in self.estimators]
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = np.asarray(X).shape[1]
        return self

    def predict_proba(self, X):
        features, _ = _feature_and_groups(X, self.group_column)
        predictions = np.column_stack([_positive_probability(model, features) for model in self.models_])
        probability = np.clip(predictions @ self.weights_, 1e-8, 1 - 1e-8)
        return np.column_stack([1 - probability, probability])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [ ]:
REG_LIBRARY = [
    ("ols", LinearRegression()),
    ("lasso", Pipeline([
        ("scale", StandardScaler()),
        ("model", LassoCV(cv=3, max_iter=5_000, n_jobs=1, random_state=SEED)),
    ])),
    ("ridge", Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 7), cv=3)),
    ])),
    ("elastic_net", Pipeline([
        ("scale", StandardScaler()),
        ("model", ElasticNetCV(cv=3, l1_ratio=[0.25, 0.5, 0.75], max_iter=5_000, n_jobs=1, random_state=SEED)),
    ])),
    ("random_forest", RandomForestRegressor(
        n_estimators=250, max_depth=15, min_samples_leaf=5, random_state=SEED, n_jobs=1,
    )),
    ("xgboost", XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=SEED, n_jobs=1, eval_metric="rmse",
    )),
]

C_GRID = np.logspace(-3, 3, 7)
CLF_LIBRARY = [
    ("logit", LogisticRegression(penalty=None, solver="lbfgs", max_iter=2_000)),
    ("lasso_logit", LogisticRegressionCV(
        Cs=C_GRID, cv=3, penalty="l1", solver="liblinear", scoring="neg_log_loss",
        max_iter=2_000, n_jobs=1, random_state=SEED,
    )),
    ("ridge_logit", LogisticRegressionCV(
        Cs=C_GRID, cv=3, penalty="l2", solver="lbfgs", scoring="neg_log_loss",
        max_iter=2_000, n_jobs=1, random_state=SEED,
    )),
    ("elastic_net_logit", LogisticRegressionCV(
        Cs=np.logspace(-2, 2, 5), cv=3, penalty="elasticnet", solver="saga",
        l1_ratios=[0.5], scoring="neg_log_loss", max_iter=3_000, n_jobs=1,
        random_state=SEED,
    )),
    ("random_forest", RandomForestClassifier(
        n_estimators=250, max_depth=15, min_samples_leaf=5, random_state=SEED, n_jobs=1,
    )),
    ("xgboost", XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=SEED, n_jobs=1, eval_metric="logloss",
    )),
]

SL_G = ConvexSuperLearnerRegressor(REG_LIBRARY, cv=INNER_FOLDS, random_state=SEED, group_column=-1)
SL_M = ConvexSuperLearnerClassifier(CLF_LIBRARY, cv=INNER_FOLDS, random_state=SEED, group_column=-1)


## 5. PSU-grouped outer cross-fitting

The outer folds are stratified by treatment category while keeping each PSU intact. The audit verifies that no PSU or household appears in both training and test, every observation is predicted exactly once per repetition, and every training fold contains all required treatment categories.


In [ ]:
def grouped_sample_splitting(frame, treatment, n_rep, n_folds=FOLDS, seed=SEED):
    y = frame[treatment].to_numpy(dtype=int)
    groups = frame["_psu_id"].to_numpy(dtype=int)
    households = frame["_hh_id"].to_numpy(dtype=int)
    countries = frame["country_cat"].astype("string").to_numpy()
    required_levels = set(np.unique(y))

    all_smpls = []
    all_smpls_cluster = []
    audits = []

    for repetition in range(n_rep):
        selected = None
        for attempt in range(100):
            splitter = StratifiedGroupKFold(
                n_splits=n_folds,
                shuffle=True,
                random_state=seed + repetition * 1_000 + attempt,
            )
            candidate = list(splitter.split(np.zeros(len(frame)), y, groups))
            if all(set(np.unique(y[train])) == required_levels for train, _ in candidate):
                selected = candidate
                break
        if selected is None:
            raise RuntimeError("Could not construct PSU-grouped folds with all treatment categories in every training sample.")

        seen = np.zeros(len(frame), dtype=int)
        rep_clusters = []
        for fold, (train, test) in enumerate(selected):
            seen[test] += 1
            train_psu = np.unique(groups[train])
            test_psu = np.unique(groups[test])
            train_hh = np.unique(households[train])
            test_hh = np.unique(households[test])

            assert np.intersect1d(train_psu, test_psu).size == 0
            assert np.intersect1d(train_hh, test_hh).size == 0

            row = {
                "repetition": repetition,
                "fold": fold,
                "train_n": len(train),
                "test_n": len(test),
                "train_psu": len(train_psu),
                "test_psu": len(test_psu),
                "train_households": len(train_hh),
                "test_households": len(test_hh),
                "test_countries": pd.Series(countries[test]).nunique(),
            }
            for level in sorted(required_levels):
                row[f"test_share_{level}"] = np.mean(y[test] == level)
                row[f"train_n_{level}"] = np.sum(y[train] == level)
            audits.append(row)
            rep_clusters.append(([train_psu], [test_psu]))

        assert np.all(seen == 1), "Every observation must occur once in the test sample per repetition."
        all_smpls.append(selected)
        all_smpls_cluster.append(rep_clusters)

    return all_smpls, all_smpls_cluster, pd.DataFrame(audits)


## 6. Observed treatment support by country and PSU

The basic support condition for each comparison is that both **No treatment** and the treatment category are observed within the country. The stricter diagnostic requires at least two distinct PSUs in each category. We do not require every individual PSU to contain both categories.


In [ ]:
def support_by_country(df, sample_name):
    sample = df.loc[df["treat_cat"].isin(LEVELS)].copy()
    counts = (
        sample.groupby(["country_cat", "treat_cat"], dropna=False)
        .agg(observations=("_row_id", "size"), PSUs=("_psu_id", "nunique"), households=("_hh_id", "nunique"))
        .reset_index()
    )

    rows = []
    summary = []
    countries = sorted(sample["country_cat"].dropna().unique())
    for level in [1, 2, 3]:
        category_rows = []
        for country in countries:
            country_counts = counts.loc[counts["country_cat"].eq(country)].set_index("treat_cat")
            n0 = int(country_counts.loc[0, "observations"]) if 0 in country_counts.index else 0
            g0 = int(country_counts.loc[0, "PSUs"]) if 0 in country_counts.index else 0
            h0 = int(country_counts.loc[0, "households"]) if 0 in country_counts.index else 0
            nd = int(country_counts.loc[level, "observations"]) if level in country_counts.index else 0
            gd = int(country_counts.loc[level, "PSUs"]) if level in country_counts.index else 0
            hd = int(country_counts.loc[level, "households"]) if level in country_counts.index else 0
            row = {
                "sample": sample_name,
                "country": country,
                "treatment_code": level,
                "treatment_category": LEVELS[level],
                "N_no_treatment": n0,
                "PSU_no_treatment": g0,
                "households_no_treatment": h0,
                "N_category": nd,
                "PSU_category": gd,
                "households_category": hd,
                "both_categories_present": n0 > 0 and nd > 0,
                "at_least_2_PSU_each": g0 >= 2 and gd >= 2,
            }
            rows.append(row)
            category_rows.append(row)

        category_frame = pd.DataFrame(category_rows)
        level_sample = sample.loc[sample["treat_cat"].eq(level)]
        summary.append({
            "sample": sample_name,
            "treatment_category": LEVELS[level],
            "observations": len(level_sample),
            "PSUs": level_sample["_psu_id"].nunique(),
            "households": level_sample["_hh_id"].nunique(),
            "countries_with_category": level_sample["country_cat"].nunique(),
            "countries_with_category_and_no_treatment": category_frame["both_categories_present"].sum(),
            "countries_with_at_least_2_PSU_each": category_frame["at_least_2_PSU_each"].sum(),
        })

    return pd.DataFrame(rows), pd.DataFrame(summary)


hh_support_detail, hh_support_summary = support_by_country(hh, "HH")
u5_support_detail, u5_support_summary = support_by_country(u5, "U5")

support_detail = pd.concat([hh_support_detail, u5_support_detail], ignore_index=True)
support_summary = pd.concat([hh_support_summary, u5_support_summary], ignore_index=True)

support_detail.to_csv(OUT / "positivity_support_by_country.csv", index=False)
support_summary.to_csv(OUT / "positivity_support_summary.csv", index=False)
display(support_summary)


## 7. Raw out-of-fold propensities

DoubleML clips propensity predictions for score stability. Positivity diagnostics must instead use the raw out-of-fold predictions before clipping. The notebook therefore stores both versions. The clipped values are used in the AIPW score; the raw values are used in the tables and plots. Because the estimator is AIPW rather than a simple IPW estimator, no effective-sample-size statistic is used in the main positivity table.


In [ ]:
def raw_oof_binary_probabilities(learner, frame, x_cols, target, smpls, label):
    X = frame[x_cols].to_numpy(dtype=float)
    y = np.asarray(target, dtype=int)
    records = []
    weight_records = []

    for repetition, folds in enumerate(smpls):
        prediction = np.full(len(frame), np.nan)
        for fold, (train, test) in enumerate(folds):
            fitted = clone(learner).fit(X[train], y[train])
            prediction[test] = fitted.predict_proba(X[test])[:, 1]
            if hasattr(fitted, "weights_"):
                for name, weight in zip(fitted.learner_names_, fitted.weights_):
                    weight_records.append({
                        "target": label,
                        "repetition": repetition,
                        "fold": fold,
                        "learner": name,
                        "weight": weight,
                    })
        if np.isnan(prediction).any():
            raise RuntimeError(f"Incomplete OOF propensity predictions for {label}.")
        records.append(pd.DataFrame({
            "_row_id": frame["_row_id"].to_numpy(),
            "country": frame["country_cat"].to_numpy(),
            "observed_treatment": frame[target.name].to_numpy() if target.name in frame else y,
            "target": label,
            "repetition": repetition,
            "propensity_raw": prediction,
            "propensity_clipped": np.clip(prediction, TRIM, 1 - TRIM),
        }))

    return pd.concat(records, ignore_index=True), pd.DataFrame(weight_records)


def propensity_summary(oof, sample_name):
    rows = []
    for target, group in oof.groupby("target"):
        p = group["propensity_raw"]
        rows.append({
            "sample": sample_name,
            "treatment_category": target,
            "P1": p.quantile(0.01),
            "P5": p.quantile(0.05),
            "median": p.quantile(0.50),
            "P95": p.quantile(0.95),
            "P99": p.quantile(0.99),
            "below_0.01_percent": 100 * p.lt(0.01).mean(),
            "below_0.025_percent": 100 * p.lt(0.025).mean(),
            "below_0.05_percent": 100 * p.lt(0.05).mean(),
            "above_0.99_percent": 100 * p.gt(0.99).mean(),
        })
    return pd.DataFrame(rows)


def ecdf(values):
    x = np.sort(np.asarray(values))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


def plot_contrast_overlap(oof, observed, level, sample_name):
    probability = oof.loc[oof["target"].eq(LEVELS[level])].copy()
    probability["observed_category"] = np.tile(observed, probability["repetition"].nunique())
    probability = probability.loc[probability["observed_category"].isin([0, level])]

    plt.figure(figsize=(7, 4.5))
    for category in [0, level]:
        values = probability.loc[probability["observed_category"].eq(category), "propensity_raw"]
        x, y = ecdf(values)
        plt.plot(x, y, label=LEVELS[category])
    for threshold in [0.01, 0.025, 0.05]:
        plt.axvline(threshold, linestyle="--", linewidth=0.8)
    plt.xlabel(f"Raw OOF P({LEVELS[level]} | X)")
    plt.ylabel("Empirical cumulative probability")
    plt.title(f"{sample_name}: {LEVELS[level]} vs No treatment")
    plt.legend()
    plt.tight_layout()
    path = FIGS / f"positivity_ecdf_{sample_name.lower()}_{level}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


## 8. Model fitting and checkpoints

All new checkpoints are stored under `Output/models/grouped_convex_sl`. Earlier non-grouped or non-convex checkpoints are intentionally not reused. The signature includes the exact sample, controls, grouped split, treatment coding, clipping threshold, and learner definitions.


In [ ]:
def ps_config_kwargs():
    if PSProcessorConfig is not None:
        return {"ps_processor_config": PSProcessorConfig(clipping_threshold=TRIM)}
    return {"trimming_rule": "truncate", "trimming_threshold": TRIM}


def save_pickle(path, object_):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as file:
        pickle.dump(object_, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def model_signature(kind, frame, outcome, treatment, x_cols, smpls, n_rep):
    settings = {
        "kind": kind,
        "rows": joblib_hash(frame[["_row_id", outcome, treatment, "_psu_id", "_hh_id"] + x_cols]),
        "outcome": outcome,
        "treatment": treatment,
        "x_cols": x_cols,
        "splits": joblib_hash(smpls),
        "n_rep": n_rep,
        "trim": TRIM,
        "levels": LEVELS,
        "ml_g": joblib_hash(SL_G),
        "ml_m": joblib_hash(SL_M),
        "doubleml": dml.__version__,
        "sklearn": sklearn.__version__,
    }
    return joblib_hash(settings)


def fit_irm(frame, x_cols, outcome, treatment, n_rep, cache_path):
    smpls, smpls_cluster, audit = grouped_sample_splitting(frame, treatment, n_rep=n_rep)
    signature = model_signature("IRM", frame, outcome, treatment, x_cols, smpls, n_rep)

    if cache_path.is_file():
        with cache_path.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("signature") == signature:
            return cached["model"], audit, smpls

    data = dml.DoubleMLData(
        frame,
        y_col=outcome,
        d_cols=treatment,
        x_cols=x_cols,
        cluster_cols="_psu_id",
    )
    model = dml.DoubleMLIRM(
        data,
        ml_g=clone(SL_G),
        ml_m=clone(SL_M),
        n_folds=FOLDS,
        n_rep=n_rep,
        score="ATE",
        draw_sample_splitting=False,
        **ps_config_kwargs(),
    )
    model.set_sample_splitting(smpls, smpls_cluster)
    model.fit(n_jobs_cv=min(WORKERS, FOLDS), store_predictions=True, store_models=False)
    save_pickle(cache_path, {"signature": signature, "model": model})
    return model, audit, smpls


def fit_apos(frame, x_cols, outcome, treatment, n_rep, cache_path):
    smpls, smpls_cluster, audit = grouped_sample_splitting(frame, treatment, n_rep=n_rep)
    signature = model_signature("APOS", frame, outcome, treatment, x_cols, smpls, n_rep)

    if cache_path.is_file():
        with cache_path.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("signature") == signature:
            return cached["model"], cached["contrast"], audit, smpls

    data = dml.DoubleMLData(
        frame,
        y_col=outcome,
        d_cols=treatment,
        x_cols=x_cols,
        cluster_cols="_psu_id",
    )
    model = dml.DoubleMLAPOS(
        data,
        ml_g=clone(SL_G),
        ml_m=clone(SL_M),
        treatment_levels=list(LEVELS),
        n_folds=FOLDS,
        n_rep=n_rep,
        draw_sample_splitting=False,
        **ps_config_kwargs(),
    )
    model.set_sample_splitting(smpls, smpls_cluster)
    model.fit(
        n_jobs_models=min(WORKERS, len(LEVELS)),
        n_jobs_cv=min(WORKERS, FOLDS),
        store_predictions=True,
        store_models=False,
    )
    contrast = model.causal_contrast(reference_levels=[0])
    save_pickle(cache_path, {"signature": signature, "model": model, "contrast": contrast})
    return model, contrast, audit, smpls


In [ ]:
def result_from_irm(model, dataset, outcome):
    summary = model.summary.iloc[0]
    return {
        "method": "IRM",
        "dataset": dataset,
        "outcome": outcome,
        "comparison": "Any recognized treatment vs No treatment",
        "coef": summary["coef"],
        "se": summary["std err"],
        "ci_low": summary["2.5 %"],
        "ci_high": summary["97.5 %"],
    }


def results_from_contrast(contrast, dataset, outcome):
    rows = []
    for index, summary in contrast.summary.iterrows():
        level = int(str(index).split()[0])
        rows.append({
            "method": "APOS contrast",
            "dataset": dataset,
            "outcome": outcome,
            "comparison": f"{LEVELS[level]} vs No treatment",
            "coef": summary["coef"],
            "se": summary["std err"],
            "ci_low": summary["2.5 %"],
            "ci_high": summary["97.5 %"],
        })
    return rows


## 9. Main samples and estimates

The two household outcomes use the same complete-case sample. `VeryHighRiskHome=1` is a subset of the positive `SomeRiskHome` cases, but it is not estimated on a restricted subsample.


In [ ]:
hh_some_irm, hh_some_irm_x = analysis_frame(hh, "SomeRiskHome", "water_treatment", child=False)
hh_vhigh_irm, hh_vhigh_irm_x = analysis_frame(hh, "VeryHighRiskHome", "water_treatment", child=False)
hh_some_apos, hh_some_apos_x = analysis_frame(hh, "SomeRiskHome", "treat_cat", child=False, allowed_levels=list(LEVELS))
hh_vhigh_apos, hh_vhigh_apos_x = analysis_frame(hh, "VeryHighRiskHome", "treat_cat", child=False, allowed_levels=list(LEVELS))
u5_irm, u5_irm_x = analysis_frame(u5, "diarrhea", "water_treatment", child=True)
u5_apos, u5_apos_x = analysis_frame(u5, "diarrhea", "treat_cat", child=True, allowed_levels=list(LEVELS))

assert hh_some_irm["_row_id"].equals(hh_vhigh_irm["_row_id"]), "HH outcome samples differ unexpectedly."
assert hh_some_apos["_row_id"].equals(hh_vhigh_apos["_row_id"]), "HH APOS outcome samples differ unexpectedly."

sample_checks = pd.DataFrame([
    {"model_sample": "HH IRM", "N": len(hh_some_irm), "PSUs": hh_some_irm["_psu_id"].nunique()},
    {"model_sample": "HH APOS", "N": len(hh_some_apos), "PSUs": hh_some_apos["_psu_id"].nunique()},
    {"model_sample": "U5 IRM", "N": len(u5_irm), "PSUs": u5_irm["_psu_id"].nunique()},
    {"model_sample": "U5 APOS", "N": len(u5_apos), "PSUs": u5_apos["_psu_id"].nunique()},
])
display(sample_checks)


In [ ]:
MAIN_SPECS = [
    ("HH", "SomeRiskHome", hh_some_irm, hh_some_irm_x, hh_some_apos, hh_some_apos_x),
    ("HH", "VeryHighRiskHome", hh_vhigh_irm, hh_vhigh_irm_x, hh_vhigh_apos, hh_vhigh_apos_x),
    ("U5", "diarrhea", u5_irm, u5_irm_x, u5_apos, u5_apos_x),
]

main_results = []
fold_audits = []
main_objects = {}

if RUN_MAIN:
    for dataset, outcome, irm_frame, irm_x, apos_frame, apos_x in MAIN_SPECS:
        irm_path = MODELS / f"irm_{dataset}_{outcome}.pkl"
        apos_path = MODELS / f"apos_{dataset}_{outcome}.pkl"

        irm_model, irm_audit, irm_smpls = fit_irm(
            irm_frame, irm_x, outcome, "water_treatment", IRM_REPS, irm_path,
        )
        apos_model, contrast, apos_audit, apos_smpls = fit_apos(
            apos_frame, apos_x, outcome, "treat_cat", APOS_REPS, apos_path,
        )

        main_results.append(result_from_irm(irm_model, dataset, outcome))
        main_results.extend(results_from_contrast(contrast, dataset, outcome))
        fold_audits.extend([
            irm_audit.assign(dataset=dataset, outcome=outcome, model="IRM"),
            apos_audit.assign(dataset=dataset, outcome=outcome, model="APOS"),
        ])
        main_objects[(dataset, outcome, "IRM")] = (irm_model, irm_frame, irm_x, irm_smpls)
        main_objects[(dataset, outcome, "APOS")] = (apos_model, apos_frame, apos_x, apos_smpls)

    main_results = pd.DataFrame(main_results)
    fold_audits = pd.concat(fold_audits, ignore_index=True)
    main_results.to_csv(OUT / "results_main_grouped_convex_sl.csv", index=False)
    fold_audits.to_csv(OUT / "fold_balance_grouped_convex_sl.csv", index=False)
    display(main_results)
    display(fold_audits.head(10))


## 10. Positivity tables and appendix plots

The household propensity diagnostics are computed once because the two E. coli outcomes have the same treatment, covariates, and estimation sample. The U5 diagnostics are computed separately.


In [ ]:
def apos_oof_bundle(frame, x_cols, smpls, sample_name):
    all_oof = []
    all_weights = []
    observed = frame["treat_cat"].to_numpy(dtype=int)
    for level in LEVELS:
        target = pd.Series((observed == level).astype(int), name="treat_cat")
        oof, weights = raw_oof_binary_probabilities(
            SL_M, frame, x_cols, target, smpls, LEVELS[level],
        )
        oof["sample"] = sample_name
        weights["sample"] = sample_name
        weights["treatment_category"] = LEVELS[level]
        all_oof.append(oof)
        all_weights.append(weights)
    return pd.concat(all_oof, ignore_index=True), pd.concat(all_weights, ignore_index=True)


if RUN_MAIN:
    hh_apos_model, hh_apos_frame, hh_apos_x, hh_apos_smpls = main_objects[("HH", "SomeRiskHome", "APOS")]
    u5_apos_model, u5_apos_frame, u5_apos_x, u5_apos_smpls = main_objects[("U5", "diarrhea", "APOS")]

    hh_oof, hh_sl_weights = apos_oof_bundle(hh_apos_frame, hh_apos_x, hh_apos_smpls, "HH")
    u5_oof, u5_sl_weights = apos_oof_bundle(u5_apos_frame, u5_apos_x, u5_apos_smpls, "U5")

    propensity_oof = pd.concat([hh_oof, u5_oof], ignore_index=True)
    propensity_table = pd.concat([
        propensity_summary(hh_oof, "HH"),
        propensity_summary(u5_oof, "U5"),
    ], ignore_index=True)
    sl_weight_table = pd.concat([hh_sl_weights, u5_sl_weights], ignore_index=True)

    propensity_oof.to_csv(OUT / "propensity_oof_raw_and_clipped.csv", index=False)
    propensity_table.to_csv(OUT / "propensity_oof_summary.csv", index=False)
    sl_weight_table.to_csv(OUT / "super_learner_propensity_weights.csv", index=False)
    display(propensity_table)

    for sample_name, oof, frame in [
        ("HH", hh_oof, hh_apos_frame),
        ("U5", u5_oof, u5_apos_frame),
    ]:
        observed = frame["treat_cat"].to_numpy(dtype=int)
        for level in [1, 2, 3]:
            plot_contrast_overlap(oof, observed, level, sample_name)


## 11. Support-restricted robustness

For each treatment comparison, the first restriction keeps countries in which both categories are present. The stricter restriction additionally requires at least two distinct PSUs in each category. These are robustness analyses rather than replacements for the main pooled estimand.


In [ ]:
def eligible_countries(df, level, min_psu):
    detail, _ = support_by_country(df, "temporary")
    subset = detail.loc[detail["treatment_code"].eq(level)]
    if min_psu == 1:
        return set(subset.loc[subset["both_categories_present"], "country"])
    return set(subset.loc[subset["at_least_2_PSU_each"], "country"])


def support_restricted_frame(df, outcome, level, child, min_psu):
    countries = eligible_countries(df, level, min_psu=min_psu)
    sample = df.loc[
        df["country_cat"].isin(countries)
        & df["treat_cat"].isin([0, level])
    ].copy()
    sample["pair_treatment"] = sample["treat_cat"].eq(level).astype("int8")
    frame, x_cols = analysis_frame(sample, outcome, "pair_treatment", child=child, allowed_levels=[0, 1])
    return frame, x_cols, countries


support_results = []
if RUN_SUPPORT_ROBUSTNESS:
    raw_specs = [
        ("HH", hh, "SomeRiskHome", False),
        ("HH", hh, "VeryHighRiskHome", False),
        ("U5", u5, "diarrhea", True),
    ]
    for dataset, raw_df, outcome, child in raw_specs:
        for level in [1, 2, 3]:
            for rule, min_psu in [("Both categories present", 1), (">=2 PSUs in each category", 2)]:
                frame, x_cols, countries = support_restricted_frame(raw_df, outcome, level, child, min_psu)
                if frame["pair_treatment"].nunique() < 2 or frame["_psu_id"].nunique() < FOLDS:
                    continue
                path = SUPPORT_MODELS / f"{dataset}_{outcome}_{level}_{min_psu}psu.pkl"
                model, audit, _ = fit_irm(
                    frame, x_cols, outcome, "pair_treatment", ROBUSTNESS_REPS, path,
                )
                result = result_from_irm(model, dataset, outcome)
                result.update({
                    "method": "Support-restricted IRM",
                    "comparison": f"{LEVELS[level]} vs No treatment",
                    "support_rule": rule,
                    "countries": len(countries),
                    "N": len(frame),
                    "PSUs": frame["_psu_id"].nunique(),
                })
                support_results.append(result)

    support_results = pd.DataFrame(support_results)
    support_results.to_csv(OUT / "results_support_restricted.csv", index=False)
    display(support_results)


## 12. Leave-one-country-out analysis

LOCO evaluates whether the pooled estimate is driven disproportionately by one country. It is distinct from the positivity checks above. Country indicators and all grouped folds are rebuilt after each country is removed.


In [ ]:
def fit_country_exclusion(raw_df, dataset, outcome, child, excluded_country):
    restricted = raw_df.loc[~raw_df["country_cat"].eq(excluded_country)].copy()
    irm_frame, irm_x = analysis_frame(restricted, outcome, "water_treatment", child=child)
    apos_frame, apos_x = analysis_frame(restricted, outcome, "treat_cat", child=child, allowed_levels=list(LEVELS))

    if irm_frame["water_treatment"].nunique() < 2:
        return []
    if set(apos_frame["treat_cat"].unique()) != set(LEVELS):
        return []

    tag = str(excluded_country).replace("/", "_").replace(" ", "_")
    irm_model, _, _ = fit_irm(
        irm_frame, irm_x, outcome, "water_treatment", LOCO_REPS,
        LOCO_MODELS / f"irm_{dataset}_{outcome}_drop_{tag}.pkl",
    )
    apos_model, contrast, _, _ = fit_apos(
        apos_frame, apos_x, outcome, "treat_cat", LOCO_REPS,
        LOCO_MODELS / f"apos_{dataset}_{outcome}_drop_{tag}.pkl",
    )

    rows = [result_from_irm(irm_model, dataset, outcome)]
    rows.extend(results_from_contrast(contrast, dataset, outcome))
    for row in rows:
        row.update({
            "excluded_country": excluded_country,
            "remaining_N_IRM": len(irm_frame),
            "remaining_PSU_IRM": irm_frame["_psu_id"].nunique(),
            "remaining_N_APOS": len(apos_frame),
            "remaining_PSU_APOS": apos_frame["_psu_id"].nunique(),
        })
    return rows


loco_results = []
if RUN_LOCO and RUN_MAIN:
    raw_specs = [
        ("HH", hh, "SomeRiskHome", False),
        ("HH", hh, "VeryHighRiskHome", False),
        ("U5", u5, "diarrhea", True),
    ]
    for dataset, raw_df, outcome, child in raw_specs:
        countries = sorted(raw_df["country_cat"].dropna().unique())
        for country in countries:
            loco_results.extend(fit_country_exclusion(raw_df, dataset, outcome, child, country))
            pd.DataFrame(loco_results).to_csv(OUT / "results_leave_one_country_out.csv", index=False)

    loco_results = pd.DataFrame(loco_results)
    full = main_results[["dataset", "outcome", "comparison", "coef", "se"]].rename(
        columns={"coef": "full_coef", "se": "full_se"}
    )
    loco_results = loco_results.merge(full, on=["dataset", "outcome", "comparison"], how="left")
    loco_results["change_from_full"] = loco_results["coef"] - loco_results["full_coef"]
    loco_results["change_in_full_SE"] = loco_results["change_from_full"] / loco_results["full_se"]
    loco_results.to_csv(OUT / "results_leave_one_country_out.csv", index=False)
    display(loco_results.head())


In [ ]:
if RUN_LOCO and isinstance(loco_results, pd.DataFrame) and not loco_results.empty:
    for (dataset, outcome, comparison), group in loco_results.groupby(["dataset", "outcome", "comparison"]):
        plot = group.sort_values("change_in_full_SE")
        plt.figure(figsize=(8, max(4, 0.25 * len(plot))))
        plt.barh(plot["excluded_country"].astype(str), plot["change_in_full_SE"])
        plt.axvline(0, linewidth=1)
        plt.xlabel("Change from full estimate, in full-sample standard errors")
        plt.title(f"LOCO: {dataset} | {OUTCOME_LABELS[outcome]} | {comparison}")
        plt.tight_layout()
        safe = f"{dataset}_{outcome}_{comparison}".replace(" ", "_").replace("/", "_")
        plt.savefig(FIGS / f"loco_{safe}.png", dpi=300, bbox_inches="tight")
        plt.show()


## 13. Optional base-learner robustness

The following analysis is deliberately disabled by default. When enabled, it re-estimates the grouped DDML specifications with each base nuisance learner separately. These estimates belong in the appendix and should not be interpreted as independent causal estimators.


In [ ]:
def base_learner_pairs():
    regression = dict(REG_LIBRARY)
    classification = dict(CLF_LIBRARY)
    return {
        "ols_logit": (regression["ols"], classification["logit"]),
        "lasso": (regression["lasso"], classification["lasso_logit"]),
        "ridge": (regression["ridge"], classification["ridge_logit"]),
        "elastic_net": (regression["elastic_net"], classification["elastic_net_logit"]),
        "random_forest": (regression["random_forest"], classification["random_forest"]),
        "xgboost": (regression["xgboost"], classification["xgboost"]),
    }


def cluster_partition_from_smpls(frame, smpls):
    groups = frame["_psu_id"].to_numpy(dtype=int)
    return [[
        ([np.unique(groups[train])], [np.unique(groups[test])])
        for train, test in repetition
    ] for repetition in smpls]


def fit_with_fixed_learners(frame, x_cols, outcome, treatment, smpls, ml_g, ml_m, kind):
    data = dml.DoubleMLData(
        frame, y_col=outcome, d_cols=treatment, x_cols=x_cols, cluster_cols="_psu_id",
    )
    kwargs = dict(
        obj_dml_data=data,
        ml_g=clone(ml_g),
        ml_m=clone(ml_m),
        n_folds=FOLDS,
        n_rep=len(smpls),
        draw_sample_splitting=False,
        **ps_config_kwargs(),
    )
    if kind == "IRM":
        model = dml.DoubleMLIRM(score="ATE", **kwargs)
    else:
        model = dml.DoubleMLAPOS(treatment_levels=list(LEVELS), **kwargs)
    model.set_sample_splitting(smpls, cluster_partition_from_smpls(frame, smpls))
    model.fit(n_jobs_cv=min(WORKERS, FOLDS), store_predictions=False, store_models=False)
    return model


base_robustness = []
if RUN_BASE_LEARNER_ROBUSTNESS and RUN_MAIN:
    for dataset, outcome, irm_frame, irm_x, apos_frame, apos_x in MAIN_SPECS:
        irm_smpls = main_objects[(dataset, outcome, "IRM")][3]
        apos_smpls = main_objects[(dataset, outcome, "APOS")][3]
        for learner_name, (ml_g, ml_m) in base_learner_pairs().items():
            irm = fit_with_fixed_learners(
                irm_frame, irm_x, outcome, "water_treatment", irm_smpls, ml_g, ml_m, "IRM",
            )
            apos = fit_with_fixed_learners(
                apos_frame, apos_x, outcome, "treat_cat", apos_smpls, ml_g, ml_m, "APOS",
            )
            row = result_from_irm(irm, dataset, outcome)
            row["nuisance_learner"] = learner_name
            base_robustness.append(row)
            contrast = apos.causal_contrast(reference_levels=[0])
            for row in results_from_contrast(contrast, dataset, outcome):
                row["nuisance_learner"] = learner_name
                base_robustness.append(row)

    base_robustness = pd.DataFrame(base_robustness)
    base_robustness.to_csv(OUT / "appendix_nuisance_learner_robustness.csv", index=False)
    display(base_robustness)


## 14. Final audit

The files below are the definitive outputs from this specification. Old `Filter`, `Chlorine`, and `Other` labels and old checkpoints must not be merged with these results.


In [ ]:
output_manifest = {
    "main_results": str(OUT / "results_main_grouped_convex_sl.csv"),
    "PSU_structure": str(OUT / "psu_structure_summary.csv"),
    "fold_balance": str(OUT / "fold_balance_grouped_convex_sl.csv"),
    "support_summary": str(OUT / "positivity_support_summary.csv"),
    "support_by_country": str(OUT / "positivity_support_by_country.csv"),
    "raw_and_clipped_propensities": str(OUT / "propensity_oof_raw_and_clipped.csv"),
    "propensity_summary": str(OUT / "propensity_oof_summary.csv"),
    "support_restricted_results": str(OUT / "results_support_restricted.csv"),
    "LOCO_results": str(OUT / "results_leave_one_country_out.csv"),
}

(OUT / "grouped_convex_sl_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)
display(pd.Series(output_manifest, name="path"))
